In [ ]:
# ========================================================
# SETUP: Install Required Packages
# ========================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("Installing packages...")
# Uncomment these lines on first run:
%pip install -q "transformers==4.46.3" "accelerate>=0.21.0" 
%pip install -q easyocr xgboost scikit-learn
%pip install cython shapely pyclipper scikit-image
%pip install "paddleocr>=2.6.1"

import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
print("✓ Setup complete!")

Installing/upgrading packages...

⚠️  IMPORTANT: After this cell finishes, RESTART THE KERNEL!
   Kaggle: Runtime → Restart Session
   Then re-run ALL cells from the beginning.

Installing bitsandbytes...


/home/samim01/Code/kaggle/myvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✓ Transformers: 4.57.3
✓ PyTorch: 2.9.1+cu128
✓ Setup complete - NOW RESTART KERNEL!


In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ========================================================
# STEP 1: EasyOCR + PaddleOCR Ensemble
# ========================================================
import os
import pandas as pd
from tqdm import tqdm
import easyocr

# TRAIN_CSV = 'PoliMemeDecode/Train/Train.csv'
TRAIN_CSV = 'Train/Train.csv'
TEST_CSV = 'Test/Test.csv'
TRAIN_DIR = 'Train/Image'
TEST_DIR = 'Test/Image'

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Try EasyOCR + PaddleOCR ensemble
print("Initializing OCR engines...")
easy_reader = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available(), verbose=False)

# Try PaddleOCR
paddle_reader = None
try:
    from paddleocr import PaddleOCR
    paddle_reader = PaddleOCR(use_angle_cls=True, lang='bn', use_gpu=torch.cuda.is_available(), show_log=False)
    print("✓ Using EasyOCR + PaddleOCR ensemble")
except:
    print("✓ Using EasyOCR only (PaddleOCR unavailable)")

def extract_text_ensemble(img_path):
    """Ensemble EasyOCR + PaddleOCR for best results"""
    texts = []
    
    # EasyOCR
    try:
        result = easy_reader.readtext(img_path, detail=0, paragraph=True)
        easy_text = " ".join(result).strip() if result else ""
        texts.append(easy_text)
    except:
        pass
    
    # PaddleOCR
    if paddle_reader:
        try:
            result = paddle_reader.ocr(img_path, cls=True)
            paddle_text = " ".join([line[1][0] for page in result for line in page]).strip() if result else ""
            texts.append(paddle_text)
        except:
            pass
    
    # Return longest non-empty result
    texts = [t for t in texts if t and t.lower() != 'no text']
    return max(texts, key=len) if texts else "no text"

# Run OCR on train
print("\nRunning OCR on TRAIN images...")
train_texts = []
for img_name in tqdm(train_df['Image_name'], desc="Train OCR"):
    img_path = os.path.join(TRAIN_DIR, img_name)
    train_texts.append(extract_text_ensemble(img_path))
train_df['ocr_text'] = train_texts

# Run OCR on test
print("Running OCR on TEST images...")
test_texts = []
for img_name in tqdm(test_df['Image_name'], desc="Test OCR"):
    img_path = os.path.join(TEST_DIR, img_name)
    test_texts.append(extract_text_ensemble(img_path))
test_df['ocr_text'] = test_texts

print(f"✓ OCR complete: {len(train_df)} train, {len(test_df)} test samples")

✓ Found cached OCR results! Loading from disk...
  Loaded 2860 train and 330 test samples from cache

Raw OCR Output - First 30 Rows


,Image_name,Label,ocr_text
0,train0001.jpg,Political,হরেক রকম চোর দিয়া বানাইছো মেলা পিসি তোমার দরব...
1,train0002.jpg,NonPolitical,বর যাত্রী যাওয়ার টাইমে যখব গাড়িতে জায়গা কম পড়...
2,train0003.jpg,NonPolitical,তুমিকেখা নোবলল বুকে অনেক ব্যখাহেয় TARLBAU; ঠিক...
3,train0004.jpg,NonPolitical,যখন ভল্লুক এসে পড়ে এবং আপন এবং আপনার বেস্টফ়র...
4,train0005.jpg,NonPolitical,বন্ধুযখন তার গফকেকাজিন বলে পরিচয় দেয় [41#4 {...
5,train0006.jpg,Political,*নির্বাচনের দিন এগিয়ে আসছে* || সীমান্তের প্রতি...
6,train0007.jpg,NonPolitical,98 OEEIJE 9র 9EEJEAE CEEUAAR DEEJAEE OFEIRAII ...
7,train0008.jpg,Political,৩দন্ত্যনাকিরেইপো্কিস্তানকে দোষারোপ করাউিচিংহিয...
8,train0009.jpg,NonPolitical,"আমরহর্ট কেপ্রশ্ন করলামঃ- [নভালোবাস] মানে কি?"" ..."
9,train0010.jpg,Political,আমি আওয়ামীলীগ করি করোজোনইেরপস পসরয়েখন


In [6]:
# ========================================================
# STEP 2: Enhanced Text Cleaning (Bengali + English)
# ========================================================
import re

def clean_text_bilingual(text):
    """Enhanced cleaning for Bengali/English mixed OCR text"""
    t = str(text or '').strip()
    
    if t.lower() == 'no text' or len(t) < 3:
        return t
    
    # Basic cleaning
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'(\w)\.(\w)', r'\1. \2', t)
    t = re.sub(r'([,;:!?])(\w)', r'\1 \2', t)
    
    # Remove multiple punctuation
    t = re.sub(r'[.]{2,}', '.', t)
    t = re.sub(r'[!]{2,}', '!', t)
    t = re.sub(r'[?]{2,}', '?', t)
    
    # Bengali-specific: danda (।) and double danda (॥)
    t = re.sub(r'।+', '।', t)
    t = re.sub(r'\s+।', '।', t)
    t = re.sub(r'।(?=[^\s])', '। ', t)
    t = re.sub(r'॥+', '॥', t)
    
    # Remove OCR artifacts
    t = re.sub(r'[\n\r\t]+', ' ', t)
    t = re.sub(r'[|]+', '', t)
    t = re.sub(r'[-]{3,}', '', t)
    t = re.sub(r'[_]{2,}', '', t)
    t = re.sub(r'[\u200b-\u200f\u202a-\u202e\ufeff]', '', t)  # Zero-width chars
    
    t = t.strip()
    t = re.sub(r'\s+', ' ', t)
    return t if len(t) > 0 else "no text"

print("Cleaning text...")
train_df['final_text'] = train_df['ocr_text'].apply(clean_text_bilingual)
test_df['final_text'] = test_df['ocr_text'].apply(clean_text_bilingual)

print(f"✓ Text cleaned. Sample:")
print(f"  Raw: {train_df['ocr_text'].iloc[0][:60]}...")
print(f"  Clean: {train_df['final_text'].iloc[0][:60]}...")


STARTING TEXT CLEANING

Cleaning TRAIN text...


Train: 100%|██████████| 2860/2860 [00:00<00:00, 30645.55it/s]


Cleaning TEST text...


Test: 100%|██████████| 330/330 [00:00<00:00, 26317.58it/s]

✓ Text cleaning complete (rule-based approach)

AFTER CLEANING - First 30 Rows


,Image_name,ocr_text,final_text
0,train0001.jpg,হরেক রকম চোর দিয়া বানাইছো মেলা পিসি তোমার দরব...,হরেক রকম চোর দিয়া বানাইছো মেলা পিসি তোমার দরব...
1,train0002.jpg,বর যাত্রী যাওয়ার টাইমে যখব গাড়িতে জায়গা কম পড়...,বর যাত্রী যাওয়ার টাইমে যখব গাড়িতে জায়গা কম পড়...
2,train0003.jpg,তুমিকেখা নোবলল বুকে অনেক ব্যখাহেয় TARLBAU; ঠিক...,তুমিকেখা নোবলল বুকে অনেক ব্যখাহেয় TARLBAU; ঠিক...
3,train0004.jpg,যখন ভল্লুক এসে পড়ে এবং আপন এবং আপনার বেস্টফ়র...,যখন ভল্লুক এসে পড়ে এবং আপন এবং আপনার বেস্টফ়র...
4,train0005.jpg,বন্ধুযখন তার গফকেকাজিন বলে পরিচয় দেয় [41#4 {...,বন্ধুযখন তার গফকেকাজিন বলে পরিচয় দেয় [41#4 {...
5,train0006.jpg,*নির্বাচনের দিন এগিয়ে আসছে* || সীমান্তের প্রতি...,*নির্বাচনের দিন এগিয়ে আসছে* সীমান্তের প্রতিটি ...
6,train0007.jpg,98 OEEIJE 9র 9EEJEAE CEEUAAR DEEJAEE OFEIRAII ...,98 OEEIJE 9র 9EEJEAE CEEUAAR DEEJAEE OFEIRAII ...
7,train0008.jpg,৩দন্ত্যনাকিরেইপো্কিস্তানকে দোষারোপ করাউিচিংহিয...,৩দন্ত্যনাকিরেইপো্কিস্তানকে দোষারোপ করাউিচিংহিয...
8,train0009.jpg,"আমরহর্ট কেপ্রশ্ন করলামঃ- [নভালোবাস] মানে কি?"" ...","আমরহর্ট কেপ্রশ্ন করলামঃ- [নভালোবাস] মানে কি?"" ..."
9,train0010.jpg,আমি আওয়ামীলীগ করি করোজোনইেরপস পসরয়েখন,আমি আওয়ামীলীগ করি করোজোনইেরপস পসরয়েখন


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting BanglaBERT Training...


Step,Training Loss
500,0.420300
1000,0.281400
1500,0.233800
2000,0.114000
2500,0.094300
3000,0.051900
3500,0.028100
4000,0.010300
4500,0.006800
5000,0.013700


Done! Text-only submission saved.


In [ ]:
# ========================================================
# STEP 3: 3-Fold CV Text Model (BanglaBERT Large)
# ========================================================
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import gc

TEXT_MODEL = "csebuetnlp/banglabert_large"
print(f"Loading {TEXT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=384):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), truncation=True, padding='max_length', 
                       max_length=self.max_len, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

labels = train_df['Label'].map({'NonPolitical': 0, 'Political': 1}).values
texts = train_df['final_text'].fillna('no text').values
texts_test = test_df['final_text'].fillna('no text').values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
text_oof = np.zeros((len(train_df), 2), dtype=np.float32)
text_test_preds = []

print("\n=== 5-Fold CV Text Training ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(texts, labels), 1):
    print(f"\nFold {fold}/5...")
    
    torch.cuda.empty_cache()
    gc.collect()
    
    model = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL, num_labels=2)
    train_ds = TextDataset(texts[tr_idx], labels[tr_idx])
    valid_ds = TextDataset(texts[va_idx], labels[va_idx])
    test_ds = TextDataset(texts_test)
    
    args = TrainingArguments(
        output_dir=f'./text_f{fold}',
        num_train_epochs=10,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=3e-5,
        weight_decay=0.01,
        fp16=True,
        report_to=[],
        evaluation_strategy='steps',
        eval_steps=100,
        save_strategy='no',
        gradient_checkpointing=True,
        logging_steps=50
    )
    
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds)
    trainer.train()
    
    text_oof[va_idx] = trainer.predict(valid_ds).predictions
    text_test_preds.append(trainer.predict(test_ds).predictions)
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

text_test_logits = np.mean(text_test_preds, axis=0)
print("✓ Text model training complete!")


--- Submission Label Counts ---
Label
NonPolitical    239
Political        91
Name: count, dtype: int64

--- Submission Preview ---


,Image_name,Label
0,test0001.jpg,NonPolitical
1,test0002.jpg,NonPolitical
2,test0003.jpg,NonPolitical
3,test0004.jpg,Political
4,test0005.jpg,NonPolitical


/home/samim01/Code/kaggle/submission_gemma_optimized.csv

In [ ]:
# ========================================================
# STEP 4: 3-Fold CV Vision Model (ViT-Large)
# ========================================================
from PIL import Image
from transformers import AutoImageProcessor, ViTForImageClassification

VISION_MODEL = "google/vit-large-patch16-224-in21k"
print(f"Loading {VISION_MODEL}...")
image_processor = AutoImageProcessor.from_pretrained(VISION_MODEL)

id2label = {0: "NonPolitical", 1: "Political"}
label2id = {"NonPolitical": 0, "Political": 1}

class ImgDataset(Dataset):
    def __init__(self, df, img_dir, processor, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor
        self.with_labels = with_labels
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image_name'])
        image = Image.open(img_path).convert('RGB')
        inputs = self.processor(images=image, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        if self.with_labels and 'Label' in row:
            item['labels'] = torch.tensor(label2id[row['Label']], dtype=torch.long)
        return item

labels_img = train_df['Label'].map(label2id).values
img_oof = np.zeros((len(train_df), 2), dtype=np.float32)
img_test_preds = []

print("\n=== 5-Fold CV Vision Training ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, labels_img), 1):
    print(f"\nFold {fold}/5...")
    
    torch.cuda.empty_cache()
    gc.collect()
    
    model = ViTForImageClassification.from_pretrained(VISION_MODEL, num_labels=2, 
                                                      id2label=id2label, label2id=label2id)
    
    train_ds = ImgDataset(train_df.iloc[tr_idx], TRAIN_DIR, image_processor)
    valid_ds = ImgDataset(train_df.iloc[va_idx], TRAIN_DIR, image_processor)
    test_ds = ImgDataset(test_df, TEST_DIR, image_processor, with_labels=False)
    
    args = TrainingArguments(
        output_dir=f'./vision_f{fold}',
        num_train_epochs=15,
        per_device_train_batch_size=24,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        fp16=True,
        report_to=[],
        evaluation_strategy='steps',
        eval_steps=50,
        save_strategy='no',
        remove_unused_columns=False,
        gradient_checkpointing=True,
        logging_steps=25
    )
    
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds)
    trainer.train()
    
    img_oof[va_idx] = trainer.predict(valid_ds).predictions
    img_test_preds.append(trainer.predict(test_ds).predictions)
    
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

img_test_logits = np.mean(img_test_preds, axis=0)
print("✓ Vision model training complete!")


=== Improving OCR with PaddleOCR (PP-OCRv3) ===
PaddleOCR not available: No module named 'paddleocr'
Attempting installation (using standard PyPI packages)...
Installing paddlepaddle-gpu (latest stable)...
✓ PaddleOCR installed successfully
Running OCR ensemble: EasyOCR + PaddleOCR
Paddle reader unavailable; using EasyOCR only.
Using ensembled OCR column: ocr_text_ens (falls back to EasyOCR if PaddleOCR missing)
✓ PaddleOCR installed successfully
Running OCR ensemble: EasyOCR + PaddleOCR
Paddle reader unavailable; using EasyOCR only.
Using ensembled OCR column: ocr_text_ens (falls back to EasyOCR if PaddleOCR missing)


In [ ]:
# ========================================================
# STEP 5: XGBoost Fusion with Rich Features
# ========================================================
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score

# Convert logits to probabilities
text_oof_probs = torch.softmax(torch.tensor(text_oof), dim=1).numpy()
text_test_probs = torch.softmax(torch.tensor(text_test_logits), dim=1).numpy()

img_oof_probs = torch.softmax(torch.tensor(img_oof), dim=1).numpy()
img_test_probs = torch.softmax(torch.tensor(img_test_logits), dim=1).numpy()

def create_fusion_features(text_p0, text_p1, img_p0, img_p1, df):
    """Generate 35+ rich features for XGBoost with NLP concepts"""
    # Text analysis
    texts = df['final_text'].fillna('').str.lower()
    text_lens = texts.str.len()
    word_counts = texts.str.split().str.len().fillna(0)
    
    # Bengali detection (Unicode range for Bengali: \u0980-\u09FF)
    bengali_chars = texts.str.count(r'[\u0980-\u09FF]')
    english_chars = texts.str.count(r'[a-zA-Z]')
    
    # NLP features - Political keywords (common in Bengali/English political memes)
    political_keywords_bn = ['রাজনীতি', 'নেতা', 'দল', 'সরকার', 'মন্ত্রী', 'প্রধানমন্ত্রী', 'নির্বাচন', 'ভোট']
    political_keywords_en = ['politic', 'leader', 'party', 'government', 'minister', 'prime', 'election', 'vote', 'parliament']
    
    # Count political keywords
    bn_pol_count = sum(texts.str.count(kw) for kw in political_keywords_bn)
    en_pol_count = sum(texts.str.count(kw) for kw in political_keywords_en)
    
    # Sentiment indicators (exclamation, question marks - emotional intensity)
    exclamation_count = texts.str.count(r'!')
    question_count = texts.str.count(r'\?')
    
    # All caps words (shouting/emphasis - common in memes)
    all_caps_words = texts.str.findall(r'\b[A-Z]{2,}\b').str.len().fillna(0)
    
    # Special characters density (memes often have emojis/special chars)
    special_char_count = texts.str.count(r'[^\w\s\u0980-\u09FF]')
    special_char_ratio = special_char_count / (text_lens + 1)
    
    # N-gram features: unique word ratio (lexical diversity)
    unique_word_ratio = texts.apply(lambda x: len(set(str(x).split())) / (len(str(x).split()) + 1) if len(str(x).split()) > 0 else 0)
    
    # Average word length (political text tends to have longer words)
    avg_word_len = texts.apply(lambda x: np.mean([len(w) for w in str(x).split()]) if len(str(x).split()) > 0 else 0)
    
    return pd.DataFrame({
        # Base probabilities (4)
        'text_p1': text_p1,
        'text_p0': text_p0,
        'img_p1': img_p1,
        'img_p0': img_p0,
        
        # Confidence scores (2)
        'text_conf': np.abs(text_p1 - 0.5) * 2,
        'img_conf': np.abs(img_p1 - 0.5) * 2,
        
        # Agreement features (4)
        'avg_p1': (text_p1 + img_p1) / 2,
        'max_p1': np.maximum(text_p1, img_p1),
        'min_p1': np.minimum(text_p1, img_p1),
        'diff_p1': np.abs(text_p1 - img_p1),
        
        # Interaction features (4)
        'text_img_prod': text_p1 * img_p1,
        'text_img_prod_p0': text_p0 * img_p0,
        'weighted_avg': text_p1 * 0.6 + img_p1 * 0.4,
        'conf_weighted': (text_p1 * np.abs(text_p1-0.5) + img_p1 * np.abs(img_p1-0.5)) / (np.abs(text_p1-0.5) + np.abs(img_p1-0.5) + 1e-8),
        
        # Text metadata (3)
        'text_len': text_lens,
        'text_word_count': word_counts,
        'has_text': (texts != 'no text').astype(int),
        
        # OCR quality indicators (4)
        'bengali_char_count': bengali_chars,
        'english_char_count': english_chars,
        'bengali_ratio': bengali_chars / (text_lens + 1),
        'lang_mix': ((bengali_chars > 0) & (english_chars > 0)).astype(int),
        
        # Model disagreement patterns (3)
        'both_uncertain': ((np.abs(text_p1-0.5) < 0.2) & (np.abs(img_p1-0.5) < 0.2)).astype(int),
        'diff_squared': (text_p1 - img_p1) ** 2,
        'conf_ratio': (np.abs(text_p1-0.5) + 1e-8) / (np.abs(img_p1-0.5) + 1e-8),
        
        # Higher-order interactions (2)
        'text_p1_squared': text_p1 ** 2,
        'img_p1_squared': img_p1 ** 2,
        
        # NLP features - Political content (4)
        'bengali_political_kw': bn_pol_count,
        'english_political_kw': en_pol_count,
        'total_political_kw': bn_pol_count + en_pol_count,
        'has_political_kw': ((bn_pol_count + en_pol_count) > 0).astype(int),
        
        # NLP features - Sentiment/Emotion (4)
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'emotional_intensity': exclamation_count + question_count,
        'all_caps_words': all_caps_words,
        
        # NLP features - Text characteristics (4)
        'special_char_ratio': special_char_ratio,
        'unique_word_ratio': unique_word_ratio,
        'avg_word_length': avg_word_len,
        'text_density': word_counts / (text_lens + 1),  # words per character
    })

print("\n=== XGBoost Fusion Training ===")
X_train = create_fusion_features(text_oof_probs[:,0], text_oof_probs[:,1], 
                                  img_oof_probs[:,0], img_oof_probs[:,1], train_df)
y_train = train_df['Label'].map(label2id).values

X_test = create_fusion_features(text_test_probs[:,0], text_test_probs[:,1], 
                                 img_test_probs[:,0], img_test_probs[:,1], test_df)

print(f"Training XGBoost with {X_train.shape[1]} features...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

xgb_model.fit(X_train, y_train, verbose=False)

# Evaluate
oof_proba = xgb_model.predict_proba(X_train)[:,1]
print(f"\n✓ OOF AUC: {roc_auc_score(y_train, oof_proba):.4f}")
print(f"✓ OOF Accuracy: {accuracy_score(y_train, (oof_proba >= 0.5).astype(int)):.4f}")

# Feature importance
feature_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 Features:")
print(feature_imp.head(10).to_string(index=False))

# Generate final submission
test_proba = xgb_model.predict_proba(X_test)[:,1]
submission = pd.DataFrame({
    'Image_name': test_df['Image_name'],
    'Label': ['Political' if p >= 0.5 else 'NonPolitical' for p in test_proba]
})

submission.to_csv('sm/submission.csv', index=False)
print(f"\n✓ FINAL SUBMISSION SAVED: submission.csv")
print(f"  Political: {(submission['Label'] == 'Political').sum()}")

print(f"  NonPolitical: {(submission['Label'] == 'NonPolitical').sum()}")
print(f"   No decisions made on text or image alone - only combined analysis.")

print(f"\n💡 This submission uses BOTH text AND image context via XGBoost fusion.")


=== Cleaning text with ensembled OCR (Bengali + English support) ===
Using text column: ocr_text_ens
Cleaning TRAIN (Bengali + English)...


clean-train: 100%|██████████| 2860/2860 [00:00<00:00, 24941.02it/s]


Cleaning TEST (Bengali + English)...


clean-test: 100%|██████████| 330/330 [00:00<00:00, 25301.99it/s]

✓ Saved: train_cleaned_ens.csv, test_cleaned_ens.csv

--- Cleaned Text Sample (first 5 rows) ---

1. Original: হরেক রকম চোর দিয়া বানাইছো মেলা পিসি তোমার দরবারে এখন ৫Bl এর খেলা I Wiewiton dai...
   Cleaned:  হরেক রকম চোর দিয়া বানাইছো মেলা পিসি তোমার দরবারে এখন ৫Bl এর খেলা I Wiewiton dai...

2. Original: বর যাত্রী যাওয়ার টাইমে যখব গাড়িতে জায়গা কম পড়়ে ছোটরাঃ...
   Cleaned:  বর যাত্রী যাওয়ার টাইমে যখব গাড়িতে জায়গা কম পড়়ে ছোটরাঃ...

3. Original: তুমিকেখা নোবলল বুকে অনেক ব্যখাহেয় TARLBAU; ঠিক মত পানি খাও এটা গ্যাস্রিকের ব্যথা...
   Cleaned:  তুমিকেখা নোবলল বুকে অনেক ব্যখাহেয় TARLBAU; ঠিক মত পানি খাও এটা গ্যাস্রিকের ব্যথা...

4. Original: যখন ভল্লুক এসে পড়ে এবং আপন এবং আপনার বেস্টফ়রেন্ড দুরজনের কেউই জানেন না কিভাবে ...
   Cleaned:  যখন ভল্লুক এসে পড়ে এবং আপন এবং আপনার বেস্টফ়রেন্ড দুরজনের কেউই জানেন না কিভাবে ...

5. Original: বন্ধুযখন তার গফকেকাজিন বলে পরিচয় দেয় [41#4 {C#I43} { @uঠe ৎemes সম্পর্কবদলে গে...
   Cleaned:  বন্ধুযখন তার গফকেকাজিন বলে পরিচয় দেয় [41#4 {C#I43} { @uঠe ৎe